# Recon 2
AWS creds, k8s, sudo, internal services.

In [ ]:
import subprocess
cmds = [
    "ls -la /var/run/secrets/ 2>&1; ls -la /var/run/secrets/vivid-oidc/ 2>&1; cat /var/run/secrets/vivid-oidc/token 2>&1 | head -c 2000",
    "ls -la /var/run/secrets/kubernetes.io/serviceaccount/ 2>&1; cat /var/run/secrets/kubernetes.io/serviceaccount/token 2>&1 | head -c 2000",
    "cat /etc/sudoers.d/vivid-sudoers 2>&1; sudo -l 2>&1",
    "ls -la /home/blender /home/connect /posit /opt/vivid-resolver /mnt/namespaces /mnt/dynamic-mounts 2>&1",
    "ls -laR /tmp/vivid-blender/ 2>&1 | head -80",
    "ls -la /cloud/persistent-state/ 2>&1; find /cloud/persistent-state -maxdepth 3 2>/dev/null | head",
    "ls -la /blenderless-entrypoint.sh; cat /blenderless-entrypoint.sh 2>&1 | head -50",
]
for c in cmds:
    print(f"$ {c}")
    p = subprocess.run(c, shell=True, capture_output=True, text=True, timeout=30)
    print(p.stdout[:6000], p.stderr[:2000])
    print("---")

In [ ]:
import subprocess
script = r'''
import urllib.request
for port in [9090, 8012, 8112, 3838, 8080, 8000, 8888, 5000]:
    for scheme in ["http", "https"]:
        try:
            r = urllib.request.urlopen(scheme + "://127.0.0.1:" + str(port), timeout=3)
            print(port, scheme, r.status, r.read(500)[:300])
            break
        except Exception as e:
            pass
'''
p = subprocess.run(["python3", "-c", script], capture_output=True, text=True, timeout=60)
print(p.stdout, p.stderr)
# who listens on 9090
for c in ["ss -tlnp 2>/dev/null || netstat -tlnp 2>/dev/null", "cat /proc/net/tcp /proc/net/tcp6 2>/dev/null"]:
    q = subprocess.run(c, shell=True, capture_output=True, text=True, timeout=15)
    print("$", c)
    print(q.stdout[:3000], q.stderr[:500])

In [ ]:
import subprocess
script = r'''
import urllib.request
def get(url):
    try:
        r = urllib.request.urlopen(url, timeout=5)
        print(url, "->", r.status, r.read(2000)[:1500])
    except Exception as e:
        print(url, "FAIL:", e)
get("https://10.96.0.1/api/v1/namespaces/vivid-apps/pods")
get("https://10.96.0.1/api/v1/namespaces/default/pods")
get("https://10.96.0.1/healthz")
'''
p = subprocess.run(["python3", "-c", script], capture_output=True, text=True, timeout=90)
print(p.stdout[:8000], p.stderr[:3000])